In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [2]:
psi = pd.read_csv(f"data/GTEx_frontal_cortex_exon_PSI.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_top_MO_Qval_modules.csv")

In [3]:
psi.shape

(44607, 269)

### Add gene names to PSI data

In [4]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs = gtf_subset["attribute"].apply(extract_attributes)
attrs_df = attrs.apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)

In [5]:
# Get PSI and GTF data ready to merge on gene IDs
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

In [6]:
psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [7]:
psi_anno.shape

(44607, 270)

In [8]:
psi_anno.columns = psi_anno.columns.str.replace("-", ".")

### Calc. corr between ME and exon PSI

In [9]:
corr_df = pd.DataFrame(
    columns=["Gene"] + top_qval_mods_df['Cell_type'].tolist(), 
    index=psi_anno.index
)
corr_df['Gene'] = psi_anno['gene_name'] 

for i, row in top_qval_mods_df.iterrows():
    ctype = row['Cell_type']

    mod_df = pd.read_csv(row['ME_path'])
    mod_eig = mod_df.set_index("Sample")[row['Module']]
    mod_eig = pd.to_numeric(mod_eig, errors="coerce")
    
    corrs = psi_anno.iloc[:, 1:].corrwith(mod_eig, axis=1)
    corr_df[ctype] = corrs

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/numpy/lib/

In [13]:
corr_df.sort_values("YANG_PFC_2021_VIP_INTERNEURON", ascending=False).head(n=10)

,Gene,YANG_PFC_2021_OLIGODENDROCYTE,YANG_PFC_2021_ASTROCYTE,YANG_PFC_2021_ENDOTHELIAL,YANG_PFC_2021_MICROGLIA,YANG_PFC_2021_L2_3_EXCITATORY_NEURON,YANG_PFC_2021_NRGN_NEURON,YANG_PFC_2021_VIP_INTERNEURON,BARRES_ASTROCYTES,HICKMAN_MICROGLIA_SENSOME_99,...,ABA_CHOROID_PLEXUS,ZEISEL_EPENDYMAL,SCHIRMER_2019_IN_VIP,SCHIRMER_2019_IN_SV2C,SCHIRMER_2019_OPC,SUGINO_UP_LAYERS4-6_GABAERGIC_NEURONS_CINGULATE_CTX,SCHIRMER_2019_IN_PVALB,SCHIRMER_2019_IN_SST,SCHIRMER_2019_EX_PYR,SCHIRMER_2019_MICROGLIA
ENSG00000231764_other_8,DLX6-AS1,-0.177538,-0.198585,-0.112975,-0.004348,0.357092,0.300706,0.357092,-0.193593,-0.010353,...,-0.003100,0.124425,0.222304,0.074590,-0.124170,0.293162,0.294459,0.167917,0.178213,-0.132936
ENSG00000139737_ProteinCoding_1,SLAIN1,-0.235947,-0.130677,-0.129959,-0.036617,0.350530,0.268412,0.350530,-0.129296,-0.036261,...,-0.069362,0.149261,0.147843,0.071114,-0.173476,0.213276,0.271730,0.190723,0.137404,-0.188496
ENSG00000231764_other_4,DLX6-AS1,-0.201433,-0.185240,-0.116912,0.032722,0.346297,0.289313,0.346297,-0.180662,0.028791,...,-0.027279,0.162415,0.180061,0.064177,-0.113025,0.261823,0.285823,0.134341,0.149049,-0.143559
ENSG00000189056_ProteinCoding_2,RELN,-0.301094,0.011994,-0.140058,-0.010456,0.343332,0.181702,0.343332,0.017099,-0.015744,...,-0.045304,0.113781,-0.018537,-0.105475,-0.201066,0.197013,0.156044,-0.003644,-0.035540,-0.186734
ENSG00000165476_NMD_1,REEP3,-0.126897,-0.192174,-0.208595,-0.122724,0.329992,0.290401,0.329992,-0.190239,-0.125648,...,-0.138788,0.060802,0.227088,0.166581,-0.226287,0.222367,0.315087,0.243983,0.250738,-0.204341
ENSG00000169855_ProteinCoding_1,ROBO1,-0.198172,-0.085095,-0.180924,-0.058529,0.324179,0.208994,0.324179,-0.085681,-0.064267,...,-0.134144,0.117641,0.106675,0.027019,-0.173508,0.183307,0.212054,0.111015,0.128354,-0.226762
ENSG00000060237_ProteinCoding_6,WNK1,-0.204421,-0.082508,-0.070076,-0.039556,0.322821,0.270196,0.322821,-0.078191,-0.040559,...,-0.042970,0.057552,0.228707,0.149914,-0.184779,0.151076,0.271416,0.234483,0.188561,-0.137341
ENSG00000185630_ProteinCoding_3,PBX1,-0.056290,-0.329969,-0.206766,-0.042318,0.317963,0.371185,0.317963,-0.327014,-0.046269,...,-0.099315,-0.041651,0.313914,0.210791,-0.201708,0.353150,0.394326,0.304561,0.272592,-0.140199
ENSG00000079805_ProteinCoding_1,DNM2,-0.159810,-0.127087,-0.178482,-0.152535,0.317521,0.223927,0.317521,-0.127121,-0.154177,...,-0.135630,0.147168,0.082414,-0.039537,-0.210737,0.252832,0.211825,0.088708,0.078396,-0.219119
ENSG00000231764_other_3,DLX6-AS1,-0.166205,-0.161709,-0.162385,0.025627,0.317231,0.282414,0.317231,-0.158869,0.024228,...,-0.050524,0.178775,0.164973,0.086766,-0.115335,0.276071,0.279704,0.152613,0.159409,-0.165193


In [14]:
corr_df.to_csv(f"data/corrs/GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_top_MO_Qval_modules_exon_corr.csv")